# NYC Taxi: Processing + Model Training Pipeline

This demo notebook runs an end-to-end Spark pipeline: data generation, feature processing, model training, evaluation, and result persistence to MinIO/Hive.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, when

spark = (SparkSession.builder
    .appName('nyc-taxi-ml-training-pipeline')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.hive.metastore.uris', 'thrift://spark-infra-spark-35-metastore:9083')
    .config('spark.sql.warehouse.dir', 's3a://warehouse/spark-35')
    .enableHiveSupport()
    .getOrCreate())

spark

In [ ]:
# Synthetic taxi-like training dataset
n = 120000
base = spark.range(n)
df = (base
    .withColumn('trip_distance', rand(seed=42) * 25)
    .withColumn('passenger_count', (rand(seed=7) * 5 + 1).cast('int'))
    .withColumn('hour', (rand(seed=9) * 24).cast('int'))
    .withColumn('is_peak', when((col('hour') >= 7) & (col('hour') <= 10), 1).otherwise(0))
    .withColumn('fare_amount',
        3.0 + col('trip_distance') * 2.2 + col('passenger_count') * 0.9 + col('is_peak') * 4.5 + rand(seed=17) * 2.0
    ))

df.select('trip_distance', 'passenger_count', 'hour', 'is_peak', 'fare_amount').show(5, truncate=False)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor

features = ['trip_distance', 'passenger_count', 'hour', 'is_peak']
assembler = VectorAssembler(inputCols=features, outputCol='features')
train_df = assembler.transform(df).select('features', col('fare_amount').alias('label'))

train, test = train_df.randomSplit([0.8, 0.2], seed=123)
model = GBTRegressor(maxIter=30, maxDepth=5, stepSize=0.1).fit(train)
pred = model.transform(test)

rmse = RegressionEvaluator(metricName='rmse').evaluate(pred)
r2 = RegressionEvaluator(metricName='r2').evaluate(pred)
print('RMSE:', round(rmse, 4))
print('R2:', round(r2, 4))

In [ ]:
from pyspark.sql.functions import lit

metrics = spark.createDataFrame([(float(rmse), float(r2), int(n))], ['rmse', 'r2', 'rows'])
metrics = metrics.withColumn('run_ts', lit(str(__import__('datetime').datetime.utcnow())))

metrics.write.mode('append').parquet('s3a://spark-jobs/model-metrics/nyc-taxi-gbt/')

spark.sql("CREATE DATABASE IF NOT EXISTS demo_shared LOCATION 's3a://warehouse/spark-35/demo_shared.db'")
metrics.createOrReplaceTempView('m')
spark.sql("CREATE TABLE IF NOT EXISTS demo_shared.nyc_taxi_training_metrics USING PARQUET LOCATION 's3a://warehouse/spark-35/demo_shared.db/nyc_taxi_training_metrics' AS SELECT * FROM m")
spark.sql("SELECT * FROM demo_shared.nyc_taxi_training_metrics ORDER BY run_ts DESC").show(10, truncate=False)